# Hadamard with ALC tone — modulated drive at +|η| for active leakage cancellation

Same machinery as `hadamard_spectral_proxy_constraint.ipynb` but with an additional drive: a slow ALC envelope `Ω_mod(t)` modulated by `exp(+i·|η|·t)`. The total rotating-frame envelope is

$$\Omega_\mathrm{total}(t) = \Omega_\mathrm{main}(t) + \Omega_\mathrm{mod}(t)\,e^{+i|\eta|t}.$$

Why: DC content of `Ω_mod` lands at `+|η|` in the spectrum of `Ω_total` — the resonant leakage sideband — and provides a *direct* knob for destructive interference with the residual leakage from the main pulse. Since reaching the `+|η|` Fourier coefficient through a finite-knot main spline is structurally expensive, having an explicit carrier-shifted channel gives the optimizer better-conditioned coordinates for the cancellation.

**Controls (4 channels)**: `u_X, u_Y, u_X_m, u_Y_m`.

**Hamiltonian (2-lvl, rotating frame at ω_10)**:
$$H(u,t) = [u_X + u_{X,m}\cos(|\eta|t) - u_{Y,m}\sin(|\eta|t)]\,\sigma_x + [u_Y + u_{X,m}\sin(|\eta|t) + u_{Y,m}\cos(|\eta|t)]\,\sigma_y.$$

**Combined-amplitude constraint** at every knot (the actual physical drive bound):
$$(u_X + u_{X,m}\cos(|\eta|t_k) - u_{Y,m}\sin(|\eta|t_k))^2 \leq a_\mathrm{bound}^2$$
$$(u_Y + u_{X,m}\sin(|\eta|t_k) + u_{Y,m}\cos(|\eta|t_k))^2 \leq a_\mathrm{bound}^2$$

Other constraints/objectives are kept identical to `hadamard_spectral_proxy_constraint.ipynb`.


In [1]:
import Pkg
Pkg.activate(@__DIR__)
piccolo_path       = joinpath(@__DIR__, "..", "..", "..", "Piccolo.jl")
directtrajopt_path = joinpath(@__DIR__, "..", "..", "..", "DirectTrajOpt.jl")
Pkg.develop([
    Pkg.PackageSpec(path = piccolo_path),
    Pkg.PackageSpec(path = directtrajopt_path),
])
Pkg.add(["CairoMakie", "MathTeXEngine", "LaTeXStrings", "JLD2",
         "DataInterpolations", "Ipopt", "QuantumToolbox", "Statistics", "FFTW", "ForwardDiff"])
Pkg.instantiate()

using Piccolo
using LinearAlgebra, Random, Printf, Statistics, JLD2
using CairoMakie, MathTeXEngine, LaTeXStrings
using DataInterpolations: CubicHermiteSpline
using SparseArrays
using FFTW

import DirectTrajOpt.Objectives: AbstractObjective, objective_value, gradient!, hessian_structure, get_full_hessian
import DirectTrajOpt.Constraints: AbstractNonlinearConstraint
import DirectTrajOpt.CommonInterface
# TrajectoryIndexingUtils is bundled via Piccolo; we inline the formula `dim*(k-1) + pos` below.
# get_times, get_timesteps come via `using Piccolo` (which re-exports NamedTrajectories).

set_theme!(Theme(
    fonts = (;
        regular = texfont(:text), bold = texfont(:bold),
        italic = texfont(:italic), bold_italic = texfont(:bolditalic),
        ticks = "TeX Gyre Heros Makie",
    ),
    Axis = (; xgridvisible = false),
))

  Activating project at `~/Library/Mobile Documents/com~apple~CloudDocs/Documents/GitHub/robust_control_sam/src/hadamard_leakage_tradeoff`
   Resolving package versions...
  No Changes to `~/Library/Mobile Documents/com~apple~CloudDocs/Documents/GitHub/robust_control_sam/src/hadamard_leakage_tradeoff/Project.toml`
  No Changes to `~/Library/Mobile Documents/com~apple~CloudDocs/Documents/GitHub/robust_control_sam/src/hadamard_leakage_tradeoff/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Library/Mobile Documents/com~apple~CloudDocs/Documents/GitHub/robust_control_sam/src/hadamard_leakage_tradeoff/Project.toml`
  No Changes to `~/Library/Mobile Documents/com~apple~CloudDocs/Documents/GitHub/robust_control_sam/src/hadamard_leakage_tradeoff/Manifest.toml`


## Parameters & operators

In [ ]:
# ---- Gate / system parameters (same as hadamard_spectral_proxy_constraint.ipynb) ----
const T_NS        = 50.0
const a_bound     = 2π * 0.01            # 10 MHz physical drive bound (applied to COMBINED signal)
const η_anh       = -2π * 0.170          # -170 MHz anharmonicity (transmon)
const δ_ALC       = η_anh
const F_THRESHOLD = 0.9999
const N_KNOTS     = 10
const Δt_GATE     = T_NS / (N_KNOTS - 1)
const Q_R_DEFAULT = 0.0
const Q_R_ROBUST  = 1000.0
const R_DDU       = 100.0
const DDU_BOUND   = 1.0
const ε_MAX       = 0.01                 # hard constraint: |Ω̂_main(η)|² ≤ ε_MAX (on the MAIN envelope only)
const R_LEAK_BARRIER = 0.0               # small leftover penalty for gradient smoothness
const NUM_ITER    = 1000
const SEED        = 42

# ALC-specific
const ALC_FREQ    = abs(η_anh)           # modulation frequency = +|η| (rotating frame)
const ALC_BOUND   = a_bound              # per-channel optimizer bound on u_X_m, u_Y_m
                                          # (the physical bound is enforced via the combined-amplitude
                                          # path constraint — see optimize_2lvl)

# 2-level operators
const σx = ComplexF64[0  1; 1  0]
const σy = ComplexF64[0 -im; im 0]
const σz = ComplexF64[1  0; 0 -1]
const I2 = Matrix{ComplexF64}(I, 2, 2)

# 3-level operators (Duffing)
const I3     = Matrix{ComplexF64}(I, 3, 3)
const a3     = ComplexF64[0 1 0; 0 0 sqrt(2); 0 0 0]
const ad3    = adjoint(a3)
const X3     = a3 + ad3
const Y3     = im * (ad3 - a3)
const n3     = ad3 * a3
const H_anh3 = (η_anh / 2) * n3 * (n3 - I3)
const subspace_indices = [1, 2]

const U_target = (1/sqrt(2)) * ComplexF64[1.0  1.0; 1.0 -1.0]
const MHz_per_radperns = 1e3 / (2π)

@printf("T=%.1fns, η=%.1fMHz, ALC_freq=%.1fMHz, N_knots=%d, ε_max=%.3f, a_bound=%.1fMHz\n",
    T_NS, η_anh/(2π)*1e3, ALC_FREQ/(2π)*1e3, N_KNOTS, ε_MAX, a_bound*MHz_per_radperns)


T=50.0ns, η=-170.0MHz, ALC_freq=170.0MHz, N_knots=10, ε_max=0.010, a_bound=10.0MHz


## Custom `SpectralLeakageObjective` (small barrier) and `SpectralLeakageConstraint` (hard bound)

The objective is the same one used in the penalty notebook (kept here at small weight `R_LEAK_BARRIER` to smooth the gradient near the active constraint boundary). The constraint is new: it computes `|Ω̂(η)|² − ε_max` and tells Ipopt it must be ≤ 0.


In [3]:
struct SpectralLeakageObjective <: AbstractObjective
    name::Symbol                    # control variable (expected to be 2 components: u_X, u_Y)
    R::Float64                      # regularization weight
    ω::Float64                      # angular frequency to suppress (rad/ns)
    times::Vector{Float64}          # cached knot times (only valid for fixed Δt)
    Δts::Vector{Float64}            # cached timesteps
    cosωt::Vector{Float64}
    sinωt::Vector{Float64}
end

function SpectralLeakageObjective(name::Symbol, ω::Float64, R::Float64, traj::NamedTrajectory)
    @assert traj.dims[name] >= 2 "SpectralLeakageObjective expects ≥ 2 components; targets the FIRST two (main u_X, u_Y)."
    times = get_times(traj)
    Δts   = get_timesteps(traj)
    return SpectralLeakageObjective(name, R, ω,
        Vector{Float64}(times), Vector{Float64}(Δts),
        cos.(ω .* times), sin.(ω .* times))
end

function objective_value(obj::SpectralLeakageObjective, traj::NamedTrajectory)
    u = traj[obj.name]
    Re_sum = 0.0; Im_sum = 0.0
    @inbounds for k in 1:length(obj.times)
        c = obj.cosωt[k]; s = obj.sinωt[k]; Δt = obj.Δts[k]
        Re_sum += Δt * (u[1, k] * c + u[2, k] * s)
        Im_sum += Δt * (u[2, k] * c - u[1, k] * s)
    end
    return obj.R * (Re_sum^2 + Im_sum^2)
end

function gradient!(∇::AbstractVector, obj::SpectralLeakageObjective, traj::NamedTrajectory)
    u = traj[obj.name]
    N = length(obj.times)
    Re_sum = 0.0; Im_sum = 0.0
    @inbounds for k in 1:N
        c = obj.cosωt[k]; s = obj.sinωt[k]; Δt = obj.Δts[k]
        Re_sum += Δt * (u[1, k] * c + u[2, k] * s)
        Im_sum += Δt * (u[2, k] * c - u[1, k] * s)
    end
    comps = traj.components[obj.name]
    @inbounds for k in 1:N
        c = obj.cosωt[k]; s = obj.sinωt[k]; Δt = obj.Δts[k]
        idxX = traj.dim * (k - 1) + comps[1]
        idxY = traj.dim * (k - 1) + comps[2]
        # ∂L/∂u_X[k] = 2R·Δt_k·(Re_sum·c - Im_sum·s)
        # ∂L/∂u_Y[k] = 2R·Δt_k·(Re_sum·s + Im_sum·c)
        ∇[idxX] += 2 * obj.R * Δt * (Re_sum * c - Im_sum * s)
        ∇[idxY] += 2 * obj.R * Δt * (Re_sum * s + Im_sum * c)
    end
    return nothing
end

function hessian_structure(obj::SpectralLeakageObjective, traj::NamedTrajectory)
    Z_dim = traj.dim * traj.N + traj.global_dim
    structure = spzeros(Z_dim, Z_dim)
    comps = traj.components[obj.name]
    idxs = Int[]
    for k in 1:traj.N
        push!(idxs, traj.dim * (k - 1) + comps[1])
        push!(idxs, traj.dim * (k - 1) + comps[2])
    end
    for i in idxs, j in idxs
        structure[i, j] = 1.0
    end
    return structure
end

function get_full_hessian(obj::SpectralLeakageObjective, traj::NamedTrajectory)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂²L = spzeros(Z_dim, Z_dim)
    comps = traj.components[obj.name]
    # L = R · (a^T x)² + R · (b^T x)² where x = control vector restricted to (u_X, u_Y) knots.
    # Hessian = 2R · (a aᵀ + b bᵀ).
    N = traj.N
    a = zeros(Z_dim); b = zeros(Z_dim)
    for k in 1:N
        c = obj.cosωt[k]; s = obj.sinωt[k]; Δt = obj.Δts[k]
        idxX = traj.dim * (k - 1) + comps[1]
        idxY = traj.dim * (k - 1) + comps[2]
        a[idxX] += Δt * c;  a[idxY] += Δt * s
        b[idxX] -= Δt * s;  b[idxY] += Δt * c
    end
    nzs = findall(!iszero, a) ∪ findall(!iszero, b)
    for i in nzs, j in nzs
        v = 2 * obj.R * (a[i]*a[j] + b[i]*b[j])
        if v != 0.0
            ∂²L[i, j] = v
        end
    end
    return ∂²L
end

# ============================================================================
# SpectralLeakageConstraint:  |Ω̂(ω)|² ≤ ε_max  as a hard inequality
# ============================================================================
struct SpectralLeakageConstraint <: AbstractNonlinearConstraint
    name::Symbol
    ω::Float64
    ε_max::Float64
    times_t::Vector{Float64}
    Δts::Vector{Float64}
    cosωt::Vector{Float64}
    sinωt::Vector{Float64}
    dim::Int
    equality::Bool
end

function SpectralLeakageConstraint(name::Symbol, ω::Float64, ε_max::Float64,
                                   traj::NamedTrajectory)
    @assert traj.dims[name] >= 2 "SpectralLeakageConstraint expects ≥ 2 components; targets the FIRST two (main u_X, u_Y)."
    times = get_times(traj)
    Δts   = get_timesteps(traj)
    return SpectralLeakageConstraint(name, ω, ε_max,
        Vector{Float64}(times), Vector{Float64}(Δts),
        cos.(ω .* times), sin.(ω .* times),
        1, false)
end

function _ReIm_sums(c::SpectralLeakageConstraint, u::AbstractMatrix)
    Re_sum = zero(eltype(u))
    Im_sum = zero(eltype(u))
    @inbounds for k in 1:length(c.times_t)
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        Re_sum += Δt_k * (u[1, k] * cos_k + u[2, k] * sin_k)
        Im_sum += Δt_k * (u[2, k] * cos_k - u[1, k] * sin_k)
    end
    return Re_sum, Im_sum
end

function CommonInterface.evaluate!(values::AbstractVector,
                                    c::SpectralLeakageConstraint,
                                    traj::NamedTrajectory)
    u = traj[c.name]
    Re_sum, Im_sum = _ReIm_sums(c, u)
    values[1] = Re_sum^2 + Im_sum^2 - c.ε_max
    return nothing
end

function CommonInterface.eval_jacobian(c::SpectralLeakageConstraint,
                                        traj::NamedTrajectory)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂g = spzeros(c.dim, Z_dim)
    u = traj[c.name]
    Re_sum, Im_sum = _ReIm_sums(c, u)
    comps = traj.components[c.name]
    @inbounds for k in 1:traj.N
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        idxX = traj.dim * (k - 1) + comps[1]
        idxY = traj.dim * (k - 1) + comps[2]
        # ∂(|Ω̂|²)/∂u_X[k] = 2·Δt_k·(Re_sum·cos - Im_sum·sin)
        # ∂(|Ω̂|²)/∂u_Y[k] = 2·Δt_k·(Re_sum·sin + Im_sum·cos)
        ∂g[1, idxX] = 2 * Δt_k * (Re_sum * cos_k - Im_sum * sin_k)
        ∂g[1, idxY] = 2 * Δt_k * (Re_sum * sin_k + Im_sum * cos_k)
    end
    return ∂g
end

function CommonInterface.eval_hessian_of_lagrangian(c::SpectralLeakageConstraint,
                                                     traj::NamedTrajectory,
                                                     μ::AbstractVector)
    # |Ω̂|² = (aᵀx)² + (bᵀx)²  ⇒  ∇² = 2·(a aᵀ + b bᵀ)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂²g = spzeros(Z_dim, Z_dim)
    comps = traj.components[c.name]
    a = zeros(Z_dim); b = zeros(Z_dim)
    @inbounds for k in 1:traj.N
        idxX = traj.dim * (k - 1) + comps[1]
        idxY = traj.dim * (k - 1) + comps[2]
        a[idxX] = c.Δts[k] * c.cosωt[k]
        a[idxY] = c.Δts[k] * c.sinωt[k]
        b[idxX] = -c.Δts[k] * c.sinωt[k]
        b[idxY] =  c.Δts[k] * c.cosωt[k]
    end
    nzs = findall(!iszero, a) ∪ findall(!iszero, b)
    for i in nzs, j in nzs
        v = 2 * μ[1] * (a[i]*a[j] + b[i]*b[j])
        if v != 0.0
            ∂²g[i, j] = v
        end
    end
    return ∂²g
end

println("SpectralLeakageObjective + SpectralLeakageConstraint defined.")


# ============================================================================
# SpectralLeakageConstraintTotal  —  constrain |Ω̂_TOTAL(ω)|² ≤ ε_max
# ============================================================================
# Same as SpectralLeakageConstraint but the Fourier coefficient is of the
# TOTAL envelope (main + modulated ALC):
#
#   Ω_total(t) = (u_X + i u_Y) + (u_X_m + i u_Y_m) · exp(+i ω t)
#
# So |Ω̂_total(ω)|² = (S_main_Re + S_alc_Re)² + (S_main_Im + S_alc_Im)²
# where the "main" parts have cos/sin factors at each knot and the "alc"
# parts are unweighted sums (since exp(+iωt)·exp(-iωt) = 1, the ALC's
# carrier collapses into a DC sum).
#
# This is the constraint to use when there's an ALC channel — constraining
# only the main's Fourier coefficient lets the ALC channel pump leakage
# unbounded.

struct SpectralLeakageConstraintTotal <: AbstractNonlinearConstraint
    name::Symbol
    ω::Float64
    ε_max::Float64
    times_t::Vector{Float64}
    Δts::Vector{Float64}
    cosωt::Vector{Float64}
    sinωt::Vector{Float64}
    dim::Int
    equality::Bool
end

function SpectralLeakageConstraintTotal(name::Symbol, ω::Float64, ε_max::Float64,
                                         traj::NamedTrajectory)
    @assert traj.dims[name] >= 4 "SpectralLeakageConstraintTotal expects ≥ 4 components (u_X, u_Y, u_X_m, u_Y_m)."
    times = get_times(traj)
    Δts   = get_timesteps(traj)
    return SpectralLeakageConstraintTotal(name, ω, ε_max,
        Vector{Float64}(times), Vector{Float64}(Δts),
        cos.(ω .* times), sin.(ω .* times),
        1, false)
end

function _ReIm_total(c::SpectralLeakageConstraintTotal, u::AbstractMatrix)
    # u is 4 × N: u[1,:]=u_X, u[2,:]=u_Y, u[3,:]=u_X_m, u[4,:]=u_Y_m
    Re_sum = zero(eltype(u))
    Im_sum = zero(eltype(u))
    @inbounds for k in 1:length(c.times_t)
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        # Main contribution to Re/Im of Ω̂_total(ω)
        Re_sum += Δt_k * (u[1, k] * cos_k + u[2, k] * sin_k)
        Im_sum += Δt_k * (u[2, k] * cos_k - u[1, k] * sin_k)
        # ALC contribution (carrier exp(+iωt) cancels exp(-iωt) → DC)
        Re_sum += Δt_k * u[3, k]   # u_X_m → Re
        Im_sum += Δt_k * u[4, k]   # u_Y_m → Im
    end
    return Re_sum, Im_sum
end

function CommonInterface.evaluate!(values::AbstractVector,
                                    c::SpectralLeakageConstraintTotal,
                                    traj::NamedTrajectory)
    u = traj[c.name]
    Re_sum, Im_sum = _ReIm_total(c, u)
    values[1] = Re_sum^2 + Im_sum^2 - c.ε_max
    return nothing
end

function CommonInterface.eval_jacobian(c::SpectralLeakageConstraintTotal,
                                        traj::NamedTrajectory)
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂g = spzeros(c.dim, Z_dim)
    u = traj[c.name]
    Re_sum, Im_sum = _ReIm_total(c, u)
    comps = traj.components[c.name]
    @inbounds for k in 1:traj.N
        cos_k = c.cosωt[k]; sin_k = c.sinωt[k]; Δt_k = c.Δts[k]
        idxX  = traj.dim * (k - 1) + comps[1]
        idxY  = traj.dim * (k - 1) + comps[2]
        idxXm = traj.dim * (k - 1) + comps[3]
        idxYm = traj.dim * (k - 1) + comps[4]
        # ∂Re/∂u_X = Δt·cos     ∂Re/∂u_Y = Δt·sin     ∂Re/∂u_X_m = Δt     ∂Re/∂u_Y_m = 0
        # ∂Im/∂u_X = -Δt·sin    ∂Im/∂u_Y = Δt·cos     ∂Im/∂u_X_m = 0      ∂Im/∂u_Y_m = Δt
        ∂g[1, idxX]  = 2 * Δt_k * (Re_sum * cos_k - Im_sum * sin_k)
        ∂g[1, idxY]  = 2 * Δt_k * (Re_sum * sin_k + Im_sum * cos_k)
        ∂g[1, idxXm] = 2 * Δt_k * Re_sum
        ∂g[1, idxYm] = 2 * Δt_k * Im_sum
    end
    return ∂g
end

function CommonInterface.eval_hessian_of_lagrangian(c::SpectralLeakageConstraintTotal,
                                                     traj::NamedTrajectory,
                                                     μ::AbstractVector)
    # Hessian of (aᵀx)² + (bᵀx)² is 2·(aaᵀ + bbᵀ).
    Z_dim = traj.dim * traj.N + traj.global_dim
    ∂²g = spzeros(Z_dim, Z_dim)
    comps = traj.components[c.name]
    a = zeros(Z_dim); b = zeros(Z_dim)
    @inbounds for k in 1:traj.N
        Δt_k = c.Δts[k]; cos_k = c.cosωt[k]; sin_k = c.sinωt[k]
        idxX  = traj.dim * (k - 1) + comps[1]
        idxY  = traj.dim * (k - 1) + comps[2]
        idxXm = traj.dim * (k - 1) + comps[3]
        idxYm = traj.dim * (k - 1) + comps[4]
        a[idxX]  =  Δt_k * cos_k
        a[idxY]  =  Δt_k * sin_k
        a[idxXm] =  Δt_k
        b[idxX]  = -Δt_k * sin_k
        b[idxY]  =  Δt_k * cos_k
        b[idxYm] =  Δt_k
    end
    nzs = findall(!iszero, a) ∪ findall(!iszero, b)
    for i in nzs, j in nzs
        v = 2 * μ[1] * (a[i]*a[j] + b[i]*b[j])
        if v != 0.0
            ∂²g[i, j] = v
        end
    end
    return ∂²g
end

println("SpectralLeakageConstraintTotal defined (constrains |Ω̂_total(ω)|² ≤ ε_max).")


# ============================================================================
# CombinedAmplitudePathConstraint  —  sub-knot bound on the COMBINED envelope
# ============================================================================
# Same idea as Piccolo's CubicHermitePathConstraint, but on the *combined* 4-channel
# envelope (with the ALC carrier modulation baked in) rather than the bare control.
# Bounds (u_X_tot)² ≤ bnd² and (u_Y_tot)² ≤ bnd² at n_samples interior sub-knot
# points per interval, using the cubic Hermite interpolation of all four channels.

using ForwardDiff

struct CombinedAmplitudePathConstraint <: AbstractNonlinearConstraint
    u_name::Symbol             # :u  (≥ 4 components)
    du_name::Symbol            # :du
    Δt_name::Symbol
    ω::Float64                 # modulation frequency (= |η|)
    bnd2::Float64              # bound²
    sample_τs::Vector{Float64} # interior fractions ∈ (0, 1)
    knot_times::Vector{Float64}
    equality::Bool
    dim::Int
    n_intervals::Int
end

function CombinedAmplitudePathConstraint(
    u_name::Symbol, ω::Float64, bnd2::Float64, traj::NamedTrajectory;
    n_samples::Int = 3, du_name::Symbol = :du, Δt_name::Symbol = traj.timestep,
)
    @assert traj.dims[u_name] >= 4 "CombinedAmplitudePathConstraint expects ≥ 4 components."
    n_intervals = traj.N - 1
    sample_τs = [j / (n_samples + 1) for j in 1:n_samples]
    knot_times = get_times(traj)
    dim = 2 * n_samples * n_intervals   # 2 residuals (X-quad, Y-quad) per sample
    return CombinedAmplitudePathConstraint(
        u_name, du_name, Δt_name, ω, bnd2,
        sample_τs, knot_times, false, dim, n_intervals,
    )
end

@inline _h00(τ) =  2τ^3 - 3τ^2 + 1
@inline _h10(τ) =   τ^3 - 2τ^2 + τ
@inline _h01(τ) = -2τ^3 + 3τ^2
@inline _h11(τ) =   τ^3 - τ^2

# z = [uₖ(4); duₖ(4); uₖ₊₁(4); duₖ₊₁(4); Δtₖ(1)]  — 17 entries
function _combined_interval(z, sample_τs, t_k, ω, bnd2)
    uXk, uYk, uXmk, uYmk         = z[1], z[2], z[3], z[4]
    duXk, duYk, duXmk, duYmk     = z[5], z[6], z[7], z[8]
    uXk1, uYk1, uXmk1, uYmk1     = z[9], z[10], z[11], z[12]
    duXk1, duYk1, duXmk1, duYmk1 = z[13], z[14], z[15], z[16]
    Δtk = z[17]

    n_s = length(sample_τs)
    out = Vector{eltype(z)}(undef, 2 * n_s)
    idx = 0
    for τ in sample_τs
        h00, h10, h01, h11 = _h00(τ), _h10(τ), _h01(τ), _h11(τ)
        uX_s  = h00*uXk  + h10*Δtk*duXk  + h01*uXk1  + h11*Δtk*duXk1
        uY_s  = h00*uYk  + h10*Δtk*duYk  + h01*uYk1  + h11*Δtk*duYk1
        uXm_s = h00*uXmk + h10*Δtk*duXmk + h01*uXmk1 + h11*Δtk*duXmk1
        uYm_s = h00*uYmk + h10*Δtk*duYmk + h01*uYmk1 + h11*Δtk*duYmk1
        t_s = t_k + τ * Δtk
        c_s, s_s = cos(ω * t_s), sin(ω * t_s)
        uX_tot = uX_s + uXm_s * c_s - uYm_s * s_s
        uY_tot = uY_s + uXm_s * s_s + uYm_s * c_s
        out[idx + 1] = uX_tot^2 - bnd2
        out[idx + 2] = uY_tot^2 - bnd2
        idx += 2
    end
    return out
end

function CommonInterface.evaluate!(
    values::AbstractVector,
    C::CombinedAmplitudePathConstraint,
    traj::NamedTrajectory,
)
    n_s = length(C.sample_τs)
    g_k_dim = 2 * n_s

    for k in 1:C.n_intervals
        uₖ    = traj[k][C.u_name]
        duₖ   = traj[k][C.du_name]
        uₖ₊₁  = traj[k+1][C.u_name]
        duₖ₊₁ = traj[k+1][C.du_name]
        Δtₖ   = traj[k][C.Δt_name][1]
        z = vcat(uₖ, duₖ, uₖ₊₁, duₖ₊₁, [Δtₖ])
        t_k = C.knot_times[k]
        values[((k-1)*g_k_dim + 1):(k*g_k_dim)] =
            _combined_interval(z, C.sample_τs, t_k, C.ω, C.bnd2)
    end
    return nothing
end

@views function CommonInterface.eval_jacobian(
    C::CombinedAmplitudePathConstraint,
    traj::NamedTrajectory,
)
    n_s = length(C.sample_τs)
    g_k_dim = 2 * n_s
    ∂C = spzeros(C.dim, traj.dim * traj.N + traj.global_dim)

    u_comps  = traj.components[C.u_name]
    du_comps = traj.components[C.du_name]
    Δt_comps = traj.components[C.Δt_name]

    for k in 1:C.n_intervals
        uₖ    = traj[k][C.u_name]
        duₖ   = traj[k][C.du_name]
        uₖ₊₁  = traj[k+1][C.u_name]
        duₖ₊₁ = traj[k+1][C.du_name]
        Δtₖ   = traj[k][C.Δt_name][1]
        z = vcat(uₖ, duₖ, uₖ₊₁, duₖ₊₁, [Δtₖ])
        t_k = C.knot_times[k]

        col_ids = vcat(
            traj.dim * (k - 1) .+ u_comps,
            traj.dim * (k - 1) .+ du_comps,
            traj.dim * k       .+ u_comps,
            traj.dim * k       .+ du_comps,
            traj.dim * (k - 1) .+ Δt_comps,
        )
        row_ids = ((k-1)*g_k_dim + 1):(k*g_k_dim)

        ∂C[row_ids, col_ids] = ForwardDiff.jacobian(
            z -> _combined_interval(z, C.sample_τs, t_k, C.ω, C.bnd2), z,
        )
    end
    return ∂C
end

@views function CommonInterface.eval_hessian_of_lagrangian(
    C::CombinedAmplitudePathConstraint,
    traj::NamedTrajectory,
    μ::AbstractVector,
)
    n_s = length(C.sample_τs)
    g_k_dim = 2 * n_s
    μ∂²C = spzeros(traj.dim * traj.N + traj.global_dim,
                    traj.dim * traj.N + traj.global_dim)

    u_comps  = traj.components[C.u_name]
    du_comps = traj.components[C.du_name]
    Δt_comps = traj.components[C.Δt_name]

    for k in 1:C.n_intervals
        uₖ    = traj[k][C.u_name]
        duₖ   = traj[k][C.du_name]
        uₖ₊₁  = traj[k+1][C.u_name]
        duₖ₊₁ = traj[k+1][C.du_name]
        Δtₖ   = traj[k][C.Δt_name][1]
        z = vcat(uₖ, duₖ, uₖ₊₁, duₖ₊₁, [Δtₖ])
        t_k = C.knot_times[k]

        col_ids = vcat(
            traj.dim * (k - 1) .+ u_comps,
            traj.dim * (k - 1) .+ du_comps,
            traj.dim * k       .+ u_comps,
            traj.dim * k       .+ du_comps,
            traj.dim * (k - 1) .+ Δt_comps,
        )

        μₖ = μ[((k-1)*g_k_dim + 1):(k*g_k_dim)]

        μ∂²C[col_ids, col_ids] += ForwardDiff.hessian(
            z -> μₖ' * _combined_interval(z, C.sample_τs, t_k, C.ω, C.bnd2), z,
        )
    end
    return μ∂²C
end

println("CombinedAmplitudePathConstraint defined (n_samples sub-knot points per interval).")


SpectralLeakageObjective + SpectralLeakageConstraint defined.
SpectralLeakageConstraintTotal defined (constrains |Ω̂_total(ω)|² ≤ ε_max).


ArgumentError: ArgumentError: Package ForwardDiff not found in current path.
- Run `import Pkg; Pkg.add("ForwardDiff")` to install the ForwardDiff package.

## Optimization — 4-channel drive with modulated ALC tone

Two variants: Default (Q_r = 0) and Robust (Q_r = 1000).
Both use:
- 4-channel `:u = (u_X, u_Y, u_X_m, u_Y_m)` — main + modulated ALC.
- Time-dependent Hamiltonian (the modulation is built into `H_fn(u, t)`).
- Spectral constraint `|Ω̂_main(|η|)|² ≤ ε_MAX` on the **first two components** (main drive only — the ALC tone is free to fill in the destructive interference contribution).
- Combined-amplitude constraint per quadrature at every knot (physical drive bound).
- Same R_ddu, Q_r, F constraint as before.


In [4]:
function optimize_2lvl(; robust::Bool, with_proxy::Bool, seed::Int = SEED)
    Random.seed!(seed)

    # 4 drive channels: u_X, u_Y, u_X_m, u_Y_m
    # Bounds: a_bound for the main, ALC_BOUND for the ALC channels.
    # The COMBINED amplitude is what gets bounded physically — that's the path constraint below.
    # The per-channel bounds here just prevent runaway.
    drive_bounds = [a_bound, a_bound, ALC_BOUND, ALC_BOUND]

    # Time-dependent Hamiltonian: main + modulated ALC
    H_fn = (u, t) -> begin
        uX, uY, uXm, uYm = u[1], u[2], u[3], u[4]
        c, s = cos(ALC_FREQ * t), sin(ALC_FREQ * t)
        uX_tot = uX + uXm * c - uYm * s
        uY_tot = uY + uXm * s + uYm * c
        return uX_tot * σx + uY_tot * σy
    end

    H_vars_fn = Function[(u, t) -> σz]
    varsys = VariationalQuantumSystem(
        H_fn, H_vars_fn, 4, drive_bounds; time_dependent = true)

    times_knots   = collect(range(0.0, T_NS, length = N_KNOTS))
    controls_init = 0.1 .* a_bound .* randn(4, N_KNOTS)
    pulse         = CubicSplinePulse(controls_init, times_knots)

    Q_r = robust ? Q_R_ROBUST : Q_R_DEFAULT

    qcp = VariationalSplinePulseProblem(
        varsys, pulse, U_target;
        Q              = 0.0,
        Q_r            = Q_r,
        R              = 1e-3,
        R_ddu          = R_DDU,
        du_bound       = Inf,
        ddu_bound      = DDU_BOUND,
        Δt_bounds      = (Δt_GATE, Δt_GATE),
        n_path_samples = 3,
    )
    push!(qcp.prob.constraints,
        FinalUnitaryFidelityConstraint(U_target, :Ũ⃗, F_THRESHOLD, get_trajectory(qcp)))

    # Spectral leakage constraint on the TOTAL envelope (main + ALC carrier-shifted).
    # CRITICAL: constraining only the main lets the ALC channel pump leakage freely
    # via its DC content (which gets carrier-shifted directly to +|η|). The total
    # constraint forces the optimizer to use the ALC channel for *destructive
    # interference* rather than free amplitude at the leakage frequency.
    if with_proxy
        traj = get_trajectory(qcp)
        spec_constr = SpectralLeakageConstraintTotal(:u, abs(η_anh), ε_MAX, traj)
        push!(qcp.prob.constraints, spec_constr)
        # Small barrier on main-envelope spectral weight, for interior-point smoothness.
        spec_obj = SpectralLeakageObjective(:u, abs(η_anh), R_LEAK_BARRIER, traj)
        qcp.prob.objective = qcp.prob.objective + spec_obj
    end

    # Combined-amplitude path constraint: enforce per-quadrature bound at
    # SUB-KNOT samples using the cubic Hermite interpolation of all 4 channels
    # plus the |η|-frequency carrier modulation evaluated at the sub-knot time.
    # n_samples = 3 → 3 interior points per interval; constraint dim = 2·3·(N-1).
    let traj = get_trajectory(qcp)
        combined_path = CombinedAmplitudePathConstraint(
            :u, ALC_FREQ, a_bound^2, traj; n_samples = 3,
        )
        push!(qcp.prob.constraints, combined_path)
    end

    label = (robust ? "ROBUST" : "DEFAULT") *
            (with_proxy ? " + SPECTRAL CONSTRAINT" : "") *
            " + ALC TONE"
    println("\n--- $label optimization ---")
    t0 = time()
    solve!(qcp; max_iter = NUM_ITER, print_level = 0,
        options = IpoptOptions(
            eval_hessian = false, constr_viol_tol = 1e-8,
            tol = 1e-8, acceptable_tol = 1e-8,
        ))
    @printf("solve wall: %.1f s\n", time() - t0)

    return get_trajectory(qcp)
end

traj_d_prox = optimize_2lvl(; robust = false, with_proxy = true)
traj_r_prox = optimize_2lvl(; robust = true,  with_proxy = true)
println("\nBoth 4-channel (main + ALC) pulses ready.")


    constructing VariationalSplinePulseProblem...
      pulse type: CubicSplinePulse{CubicHermiteSpline{Matrix{Float64}, Vector{Float64}, Vector{Union{}}, Matrix{Float64}, Vector{Vector{Float64}}, Float64}}
      time_dependent: true
      variational directions: 1
      added augmented state :var_Ũ⃗  dim=16  (1 error channels)
      dynamics spline order: 3
      added DerivativeIntegrator (:du → :ddu) for acceleration bound
      applying TimeStepsAllEqualConstraint
      added CubicHermitePathConstraint: 3 samples/interval, bound=±0.06283


UndefVarError: UndefVarError: `CombinedAmplitudePathConstraint` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Time-domain controls

In [5]:
# 4-channel spline access + combined envelope reconstruction
function spline_4channel(traj)
    us  = traj[:u]; dus = traj[:du]
    ts  = collect(range(0.0, T_NS, length = size(us, 2)))
    sp_X  = CubicHermiteSpline(dus[1, :], us[1, :], ts)
    sp_Y  = CubicHermiteSpline(dus[2, :], us[2, :], ts)
    sp_Xm = CubicHermiteSpline(dus[3, :], us[3, :], ts)
    sp_Ym = CubicHermiteSpline(dus[4, :], us[4, :], ts)
    return sp_X, sp_Y, sp_Xm, sp_Ym
end

# For backward compatibility with existing rollout helpers:
spline_uXuY(traj) = (spline_4channel(traj)[1], spline_4channel(traj)[2])

function combined_envelope(traj, t)
    sp_X, sp_Y, sp_Xm, sp_Ym = spline_4channel(traj)
    c, s = cos(ALC_FREQ * t), sin(ALC_FREQ * t)
    uX_tot = sp_X(t) + sp_Xm(t)*c - sp_Ym(t)*s
    uY_tot = sp_Y(t) + sp_Xm(t)*s + sp_Ym(t)*c
    return uX_tot, uY_tot
end

ts_fine = collect(range(0.0, T_NS, length = 4000))

function sample_4channel(traj)
    sp_X, sp_Y, sp_Xm, sp_Ym = spline_4channel(traj)
    uX  = [sp_X(t)  for t in ts_fine] .* MHz_per_radperns
    uY  = [sp_Y(t)  for t in ts_fine] .* MHz_per_radperns
    uXm = [sp_Xm(t) for t in ts_fine] .* MHz_per_radperns
    uYm = [sp_Ym(t) for t in ts_fine] .* MHz_per_radperns
    uX_tot = [combined_envelope(traj, t)[1] for t in ts_fine] .* MHz_per_radperns
    uY_tot = [combined_envelope(traj, t)[2] for t in ts_fine] .* MHz_per_radperns
    return uX, uY, uXm, uYm, uX_tot, uY_tot
end

panels = [
    ("Default + spectral + ALC", traj_d_prox, :crimson),
    ("Robust + spectral + ALC",  traj_r_prox, :forestgreen),
]

fig = Figure(size = (1500, 1000), fontsize = 13)
for (i, (lbl, traj, c)) in enumerate(panels)
    uX, uY, uXm, uYm, uX_tot, uY_tot = sample_4channel(traj)

    # Top row: main envelope u_X, u_Y
    ax_main = Axis(fig[1, i]; xlabel = "t (ns)", ylabel = "u_main (MHz)",
        title = "$lbl  —  main envelope")
    lines!(ax_main, ts_fine, uX; color = c, linewidth = 2.0, label = "u_X (main)")
    lines!(ax_main, ts_fine, uY; color = c, linewidth = 2.0, linestyle = :dot, label = "u_Y (main)")
    axislegend(ax_main; position = :rt, labelsize = 10)

    # Middle row: ALC envelope u_X_m, u_Y_m (slow)
    ax_alc = Axis(fig[2, i]; xlabel = "t (ns)", ylabel = "u_mod (MHz)",
        title = "$lbl  —  ALC slow envelope")
    lines!(ax_alc, ts_fine, uXm; color = :royalblue, linewidth = 2.0, label = "u_X_m (slow)")
    lines!(ax_alc, ts_fine, uYm; color = :royalblue, linewidth = 2.0, linestyle = :dot, label = "u_Y_m (slow)")
    axislegend(ax_alc; position = :rt, labelsize = 10)

    # Bottom row: combined envelope (what the device actually sees)
    ax_tot = Axis(fig[3, i]; xlabel = "t (ns)", ylabel = "u_total (MHz)",
        title = "$lbl  —  combined drive  (cosθ·η modulation visible)")
    lines!(ax_tot, ts_fine, uX_tot; color = :black, linewidth = 1.4, label = "u_X total")
    lines!(ax_tot, ts_fine, uY_tot; color = :black, linewidth = 1.4, linestyle = :dot, label = "u_Y total")
    hlines!(ax_tot, [a_bound, -a_bound] .* MHz_per_radperns;
        color = :gray, linestyle = :dash, linewidth = 1)
    axislegend(ax_tot; position = :rt, labelsize = 10)
end
display(fig)
save(joinpath(@__DIR__, "alc_modulated_time_domain.png"), fig; px_per_unit = 4)
fig


UndefVarError: UndefVarError: `traj_d_prox` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Frequency-domain: |Ω̂(f)|

Dashed vertical line is the leakage transition frequency f₂₁ − f₁₀ = η/(2π). The spectral proxy should push the |Ω̂| value at that frequency much lower.

In [6]:
# Spectra: main envelope alone vs total (main + modulated ALC)
function envelope_spectrum_main(traj; N_dense = 8192)
    sp_X, sp_Y, _, _ = spline_4channel(traj)
    ts_d = collect(range(0.0, T_NS, length = N_dense))
    dt_d = ts_d[2] - ts_d[1]
    Ω = [sp_X(t) + im * sp_Y(t) for t in ts_d]
    Ω̂ = fftshift(fft(Ω)) * dt_d
    fs = fftshift(fftfreq(N_dense, 1/dt_d))
    return fs .* 1e3, abs.(Ω̂)
end

function envelope_spectrum_total(traj; N_dense = 8192)
    ts_d = collect(range(0.0, T_NS, length = N_dense))
    dt_d = ts_d[2] - ts_d[1]
    Ω = [begin
        uX_tot, uY_tot = combined_envelope(traj, t)
        uX_tot + im * uY_tot
    end for t in ts_d]
    Ω̂ = fftshift(fft(Ω)) * dt_d
    fs = fftshift(fftfreq(N_dense, 1/dt_d))
    return fs .* 1e3, abs.(Ω̂)
end

η_MHz = η_anh / (2π) * 1e3

fig = Figure(size = (1500, 600), fontsize = 14)
for (i, (lbl, traj, c)) in enumerate(panels)
    fs_MHz_m, mag_m = envelope_spectrum_main(traj)
    fs_MHz_t, mag_t = envelope_spectrum_total(traj)
    Ω_at_η_m = mag_m[argmin(abs.(fs_MHz_m .- abs(η_MHz)))]
    Ω_at_η_t = mag_t[argmin(abs.(fs_MHz_t .- abs(η_MHz)))]

    ax = Axis(fig[1, i];
        xlabel = "f (MHz)", ylabel = "|Ω̂(f)|",
        yscale = log10,
        title = "$lbl\n|main(|η|)|=$(round(Ω_at_η_m,sigdigits=3))  |total(|η|)|=$(round(Ω_at_η_t,sigdigits=3))")
    lines!(ax, fs_MHz_m, max.(mag_m, 1e-10); color = c, linewidth = 2.0, label = "main")
    lines!(ax, fs_MHz_t, max.(mag_t, 1e-10); color = :black, linewidth = 1.4, linestyle = :dash, label = "total (incl. ALC)")
    vlines!(ax, [η_MHz, -η_MHz]; color = :black, linestyle = :dash, linewidth = 1.5)
    xlims!(ax, -500, 500)
    axislegend(ax; position = :rb, labelsize = 10)
end
display(fig)
save(joinpath(@__DIR__, "alc_modulated_freq_domain.png"), fig; px_per_unit = 4)
fig


UndefVarError: UndefVarError: `panels` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## 2-lvl honest verification (`unitary_rollout`, cubic-Hermite spline)

In [7]:
# Honest 2-lvl rollout: use a QuantumSystem that takes the COMBINED envelope.
# The trajectory stores 4 channels, but the actual qubit-frame drive is the
# combined envelope u_X_tot, u_Y_tot (which depends on t via the ALC modulation).
# Easiest verification: build the combined u(t) and propagate it through a 2-lvl
# ODE with σ_x, σ_y as drive operators.

using QuantumToolbox
import QuantumToolbox: Qobj, QobjEvo, sesolve, basis

const σx_qt = Qobj(σx)
const σy_qt = Qobj(σy)
const σz_qt = Qobj(σz)

function F_QT_2lvl_combined(traj; ε = 0.0)
    coef_x = (p, t) -> combined_envelope(traj, t)[1]
    coef_y = (p, t) -> combined_envelope(traj, t)[2]
    H = QobjEvo((ε * σz_qt, (σx_qt, coef_x), (σy_qt, coef_y)))
    U_T = Matrix{ComplexF64}(I, 2, 2)
    for ψ0_idx in 0:1
        ψ0 = basis(2, ψ0_idx)
        sol = sesolve(H, ψ0, [0.0, T_NS]; progress_bar = Val(false),
                      abstol = 1e-12, reltol = 1e-12)
        U_T[:, ψ0_idx + 1] = sol.states[end].data
    end
    return abs2(tr(U_target' * U_T)) / 4
end

for (lbl, traj, _) in panels
    @printf("%-30s  F_2lvl (QT, combined) = %.6f\n", lbl, F_QT_2lvl_combined(traj))
end


UndefVarError: UndefVarError: `panels` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## 2-lvl QuantumToolbox cross-check (independent ODE solver)

In [8]:
# (Cross-check moved into cell 13 — the combined envelope is what the device sees,
# so the 2-lvl honest verification needs the combined u_X_tot, u_Y_tot.)
println("OK — combined-envelope verification is in the cell above.")


OK — combined-envelope verification is in the cell above.


## 3-lvl simulation at ε = 0 (honest leakage check)

In [9]:
function propagate_3lvl_raw(traj, ε; N_fine = 2000)
    ts_f = collect(range(0.0, T_NS, length = N_fine))
    dt_sub = ts_f[2] - ts_f[1]
    U = Matrix{ComplexF64}(I, 3, 3)
    for k in 1:N_fine - 1
        t_mid = 0.5 * (ts_f[k] + ts_f[k+1])
        uX_tot, uY_tot = combined_envelope(traj, t_mid)
        H = H_anh3 + uX_tot * X3 + uY_tot * Y3 + ε * n3
        U = exp(-im * dt_sub * H) * U
    end
    return U
end

function F_L_3lvl(traj; ε = 0.0)
    U_T  = propagate_3lvl_raw(traj, ε)
    Usub = U_T[subspace_indices, subspace_indices]
    F = abs2(tr(U_target' * Usub)) / 4
    L = 1 - real(tr(Usub' * Usub)) / 2
    return F, L
end

for (lbl, traj, _) in panels
    F3, L3 = F_L_3lvl(traj)
    @printf("%-30s  F_3lvl = %.6f   L_3lvl = %.3e\n", lbl, F3, L3)
end


UndefVarError: UndefVarError: `panels` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## ε-sweep — robustness curves (2-lvl & 3-lvl)

In [10]:
εs = collect(range(-2π*0.01, 2π*0.01, length = 51))   # ±10 MHz in rad/ns
εs_MHz = εs .* MHz_per_radperns

# 2-lvl ε sweep using QT with combined envelope
function sweep_2lvl_qt(traj)
    Fs = Float64[]
    for ε in εs
        push!(Fs, F_QT_2lvl_combined(traj; ε = ε))
    end
    return Fs
end

function sweep_3lvl(traj)
    Fs = Float64[]; Ls = Float64[]
    for ε in εs
        F, L = F_L_3lvl(traj; ε = ε)
        push!(Fs, F); push!(Ls, L)
    end
    return Fs, Ls
end

println("Running 2-lvl ε-sweeps (QT, combined envelope)...")
@time F2_dp = sweep_2lvl_qt(traj_d_prox);
@time F2_rp = sweep_2lvl_qt(traj_r_prox);

println("Running 3-lvl ε-sweeps...")
@time F3_dp, L3_dp = sweep_3lvl(traj_d_prox);
@time F3_rp, L3_rp = sweep_3lvl(traj_r_prox);

println("Done.")


Running 2-lvl ε-sweeps (QT, combined envelope)...


UndefVarError: UndefVarError: `traj_d_prox` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## F vs ε (2-lvl rollout, σ_z perturbation — robustness curves)

In [11]:
fig = Figure(size = (1300, 550), fontsize = 18)
ax_F  = Axis(fig[1, 1]; xlabel = "ε (MHz)", ylabel = "F(T)",
    title = "2-lvl F vs ε (σ_z perturbation)")
ax_inf = Axis(fig[1, 2]; xlabel = "ε (MHz)", ylabel = "1 − F(T)",
    yscale = log10, title = "Infidelity, log y")

for (lbl, F, c) in [
        ("Default + spectral", F2_dp, :crimson),
        ("Robust + spectral",  F2_rp, :forestgreen),
    ]
    lines!(ax_F,   εs_MHz, F; color = c, linewidth = 2.5, label = lbl)
    lines!(ax_inf, εs_MHz, max.(1 .- F, 1e-12); color = c, linewidth = 2.5, label = lbl)
end
hlines!(ax_F, [1.0]; color = :black, linestyle = :dot, linewidth = 1)
ylims!(ax_F, 0, 1.05)
axislegend(ax_F;   position = :rb, labelsize = 12)
axislegend(ax_inf; position = :lb, labelsize = 12)
display(fig)
save(joinpath(@__DIR__, "spectral_F_vs_eps_2lvl.png"), fig; px_per_unit = 4)
fig


UndefVarError: UndefVarError: `F2_dp` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## F and leakage vs ε (3-lvl Duffing, V = n̂)

In [12]:
fig = Figure(size = (1300, 550), fontsize = 18)
ax_F = Axis(fig[1, 1]; xlabel = "ε (MHz)", ylabel = "1 − F(T)", yscale = log10, title = "3-lvl infidelity vs ε,  V = n̂")
ax_L = Axis(fig[1, 2]; xlabel = "ε (MHz)", ylabel = "leakage(T)",
    yscale = log10, title = "3-lvl leakage at gate end")

for (lbl, F, L, c) in [
        ("Default + spectral", F3_dp, L3_dp, :crimson),
        ("Robust + spectral",  F3_rp, L3_rp, :forestgreen),
    ]
    lines!(ax_F, εs_MHz, max.(1 .- F, 1e-12);
        color = c, linewidth = 2.5, label = lbl)
    lines!(ax_L, εs_MHz, max.(L, 1e-12);
        color = c, linewidth = 2.5, label = lbl)
end
axislegend(ax_F; position = :lb, labelsize = 12)
axislegend(ax_L; position = :lb, labelsize = 12)
display(fig)
save(joinpath(@__DIR__, "spectral_F_L_vs_eps_3lvl.png"), fig; px_per_unit = 4)
fig


UndefVarError: UndefVarError: `F3_dp` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Summary table

In [13]:
mid = (length(εs) + 1) ÷ 2
@printf("\n%-30s | F_2lvl(0)  | F_3lvl(0)  | L_3lvl(0)  | |Ω̂_main(η)| | |Ω̂_total(η)|\n", "pulse")
@printf("%s\n", "-"^115)
for (lbl, traj, _) in panels
    F2 = F_QT_2lvl_combined(traj)
    F3, L3 = F_L_3lvl(traj)
    fs_m, mag_m = envelope_spectrum_main(traj)
    fs_t, mag_t = envelope_spectrum_total(traj)
    Ω_m = mag_m[argmin(abs.(fs_m .- abs(η_MHz)))]
    Ω_t = mag_t[argmin(abs.(fs_t .- abs(η_MHz)))]
    @printf("%-30s | %.6f   | %.6f   | %.3e   | %.3e    | %.3e\n",
        lbl, F2, F3, L3, Ω_m, Ω_t)
end



pulse                          | F_2lvl(0)  | F_3lvl(0)  | L_3lvl(0)  | |Ω̂_main(η)| | |Ω̂_total(η)|
-------------------------------------------------------------------------------------------------------------------


UndefVarError: UndefVarError: `panels` not defined in `Main`
Suggestion: check for spelling errors or missing imports.